Last Updated : 20260227

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from polychrom.hdf5_format import list_URIs, load_URI
from polychrom.polymer_analyses import calculate_contacts
from polychrom.polymer_analyses import contact_scaling
from polychrom.contactmaps import monomerResolutionContactMap, binnedContactMap
from polychrom.polymer_analyses import slope_contact_scaling

In [2]:
#print(d.keys())

In [3]:
folder = "LEFPositions_simu_20260228_big_chromosome_2" #you can write your own trajectory

uris = list_URIs(folder)

In [ ]:


#print(len(uris))  #total_frames

#frame0 = load_URI(uris[0])
#print(frame0.keys())
#print(frame0["pos"].shape)

Ek = []
Ep = []

for uri in uris:
    d = load_URI(uri)
    Ek.append(d["kineticEnergy"])
    Ep.append(d["potentialEnergy"])

plt.plot(Ep, color = "black")
plt.grid()
plt.xlabel("Frame")
plt.ylabel("Ep [$k_{B}T$]")
plt.title("Equilibrium Checking")
plt.show()

Hi-C and P(s), $\gamma$ are recommend with only single polymer.

In [4]:
uris_eq = uris[0:] #from frame 100 to 999

In [5]:
cutoff = 1.5

In [6]:
vmin = 0
vmax = 8
#you can change the number
xlim = 3000
ylim = 3000

binSize = 4

In [ ]:
contactmap = monomerResolutionContactMap(
    uris_eq,
    cutoff=cutoff,
    n=8           #parallel processing
) / len(uris_eq) #(you can divide if you want to average...)

In [ ]:
contactmap = binnedContactMap(uris_eq, binSize = binSize, cutoff = cutoff, n = 20)

/home/hyeonjeyang/miniconda3/lib/python3.13/site-packages/polychrom/contactmaps.py:478: UserWarning: very large contact map may be difficult to visualize
  warnings.warn(UserWarning("very large contact map" " may be difficult to visualize"))


In [ ]:
print(type(contactmap))
print(len(contactmap))
print(contactmap[0])
print(contactmap[1])

In [ ]:
plt.imshow(np.log(contactmap[0]+1e-6), cmap="Reds", origin = "upper") #vmin = vmin, vmax = vmax,  
plt.title("Visualization of Hi-C Map")
plt.ylabel("distance")
plt.xlabel("distance")
plt.colorbar(label = "log(Contact frequency)")
plt.show()

In [ ]:
plt.imshow(np.log(contactmap[0][:xlim,:ylim]+1e-6), cmap="Reds", vmin = vmin, vmax = vmax, origin = "upper") #vmin = vmin, vmax = vmax,  
plt.title("Visualization of Hi-C Map")
plt.ylabel("distance")
plt.xlabel("distance")
plt.colorbar(label = "log(Contact frequency)")
plt.show()

In [ ]:
ps_list = []
for i in uris_eq:
    d = load_URI(i)
    pos = d["pos"]

    mids, ps = contact_scaling(pos, cutoff = cutoff)
    ps_list.append(ps)

ps_avg = np.mean(ps_list, axis = 0)

In [ ]:
ps_array = np.array(ps_list)   # (n_frames, n_bins)

ps_mean = ps_array.mean(axis=0)
ps_std  = ps_array.std(axis=0)
#ps_sem = ps_std / np.sqrt(ps_array.shape[0])

In [ ]:
print(contactmap[0].shape)

In [ ]:
plt.figure()

plt.loglog(mids[:30], ps_mean[:30], color="black", lw=2, label="Mean")

plt.fill_between(
    mids[:30],
    ps_mean[:30] - ps_std[:30],
    ps_mean[:30] + ps_std[:30],
    color="gray",
    alpha=0.3,
    label="±1 std"
)

plt.grid()
plt.xlabel("s")
plt.ylabel("P(s)")
plt.title("P(s)")
plt.legend()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Hi-C matrix
M = contactmap[0].copy() / 1000
N = M.shape[0]

# ---------------------------
# 1. Genomic distance별 평균 contact 계산
# ---------------------------

# distance matrix
dist = np.abs(np.subtract.outer(np.arange(N), np.arange(N)))

# 거리별 contact 합
sum_per_dist = np.bincount(dist.ravel(), weights=M.ravel())

# 거리별 원소 개수
count_per_dist = np.bincount(dist.ravel())

# 평균 contact = P(s)
Ps = sum_per_dist / count_per_dist

# s 값 (genomic distance)
s = np.arange(len(Ps))

# s=0 제외 (대각선 자기자신)
Ps = Ps[1:]
s = s[1:]

# ---------------------------
# 2. 로그-로그 플롯
# ---------------------------

plt.figure()
plt.loglog(s, Ps, label = "Mean (from Hi-C calculation)")

plt.loglog(mids[:30], ps_mean[:30], color="black", lw=2, label="Mean (from polychrom)")

plt.fill_between(
    mids[:30],
    ps_mean[:30] - ps_std[:30],
    ps_mean[:30] + ps_std[:30],
    color="gray",
    alpha=0.3,
    label="±1 std"
)

plt.grid()
plt.xlabel("s")
plt.ylabel("P(s)")
plt.title("P(s) plot")
plt.legend()
plt.show()

In [ ]:
print("Min:", M.min())
print("Max:", M.max())
print("Mean:", M.mean())

In [ ]:
mids2, slope = slope_contact_scaling(mids, ps_mean, sigma = 1)
#try to compute it in different ways
#Hi-C를 diagonal로 average해서 P(s)를 get 해보자!
#change sigma?

In [ ]:
plt.figure()
plt.plot(mids2, slope, color="black")
plt.xscale("log")
plt.xlabel("s")
plt.ylabel("$\\gamma$")
plt.title("$\\gamma$ plot")
plt.axhline(-1,  ls="--", color="gray")
plt.axhline(-1.5,ls="--", color="gray")
plt.axhline(-0.5,  ls="--", color="gray")
#plt.ylim([-2.0,0])
plt.grid()
plt.show()


In [ ]:
log_s = np.log(s)
log_Ps = np.log(Ps+1e-10)

# numerical derivative
gamma = -np.gradient(log_Ps, log_s)

plt.figure()
plt.semilogx(mids2, slope, color="black", label = "polychrom $\\gamma$")
plt.xscale("log")
plt.xlabel("s")
plt.ylabel("$\\gamma$")
plt.title("$\\gamma$ plot")
plt.axhline(-1,  ls="--", color="gray")
plt.axhline(-1.5,ls="--", color="gray")
plt.axhline(-0.5,  ls="--", color="gray")
plt.ylim([-2.0,1])
plt.grid()

plt.semilogx(s, gamma)

plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# contact_avg 는 (N x N) matrix 라고 가정
M = contactmap[0].copy()

# 대칭 보장 (혹시 수치 오차 있을 경우)
M = (M + M.T) / 2

In [ ]:
eigvals, eigvecs = np.linalg.eigh(M)

# 가장 큰 고윳값 인덱스
idx = np.argmax(eigvals)

principal_eigval = eigvals[idx]
principal_eigvec = eigvecs[:, idx]

print("Largest eigenvalue:", principal_eigval)

In [ ]:
if np.mean(principal_eigvec) < 0:
    principal_eigvec *= -1

In [ ]:
plt.figure(figsize=(10,4))
plt.plot(principal_eigvec)
plt.axhline(0, color='k', linestyle='--')
plt.title("Principal Eigenvector (Compartment Signal)")
plt.xlabel("Genomic position")
plt.ylabel("Eigenvector value")
plt.show()

In [ ]:
v = principal_eigvec

# 정규화 (선택사항 — 시각화 깔끔해짐)
v = v / np.linalg.norm(v)

compartment_matrix = np.outer(v, v)

In [ ]:
plt.figure(figsize=(6,6))
plt.imshow(compartment_matrix, cmap="coolwarm", origin="upper")
plt.colorbar(label="Compartment signal (v_i v_j)")
plt.title("Compartment Matrix (v_1)")
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# ---------------------------
# 1. Observed / Expected
# ---------------------------

N = M.shape[0]

# distance matrix
dist = np.abs(np.subtract.outer(np.arange(N), np.arange(N)))

# expected contact per genomic distance
sum_per_dist = np.bincount(dist.ravel(), weights=M.ravel())
count_per_dist = np.bincount(dist.ravel())
expected = sum_per_dist / count_per_dist

# O/E normalization
OE = M / (expected[dist] + 1e-6)

# log transform
OE_log = np.log1p(OE)

# ---------------------------
# 2. Correlation matrix
# ---------------------------

corr = np.corrcoef(OE_log)

# ---------------------------
# 3. Eigen decomposition
# ---------------------------

eigvals, eigvecs = np.linalg.eigh(corr)
idx = np.argmax(eigvals)

compartment_vec = eigvecs[:, idx]

# sign normalization
if np.mean(compartment_vec) < 0:
    compartment_vec *= -1

# ---------------------------
# 4. Compartment matrix (rank-1)
# ---------------------------

compartment_matrix = np.outer(compartment_vec, compartment_vec)

# ---------------------------
# 5. Plot
# ---------------------------

plt.figure(figsize=(14,5))

plt.subplot(1,3,1)
plt.imshow(np.log1p(M), cmap="Reds", origin="upper")
plt.title("Original Hi-C")
plt.colorbar()

plt.subplot(1,3,2)
plt.plot(compartment_vec)
plt.axhline(0, color='k', linestyle='--')
plt.title("Principal Eigenvector")
plt.xlabel("Genomic position")

plt.subplot(1,3,3)
plt.imshow(compartment_matrix, cmap="coolwarm", origin="upper")
plt.title("Compartment Matrix (Rank-1)")
plt.colorbar()

plt.tight_layout()
plt.show()